# Saltback simulator - Step 1: one storm
**How to use:** click **Runtime -> Run all**. Then move the sliders in step 2 and re-run steps 2-4.

The simulation (`simulate.py`) only produces raw numbers. A separate translator (`explain.py`)
turns them into plain English and a chart. All inputs are typical published values, not real town data.

In [ ]:
# @title 1. Load the Saltback model (click to run; no need to read the code)
open('model.py', 'w').write(r'''"""Saltback physics model - STEP 1: one storm, one stretch of road, one storm drain.

Follows the salt along its whole path:
    truck -> road (melts ice) -> runoff -> storm drain -> sensor reading -> Saltback's estimate

Everything is "per square meter of road", so results do not depend on road size.
SETTINGS holds typical published values (ASSUMPTIONS, not measured data); change them freely.
"""
import numpy as np

# ---------------- Settings you can change (typical values, with realistic ranges) ----------------
SETTINGS = dict(
    pavement_temp_F=22,       # road surface temperature during the storm               (range -5 to 32)
    ice_film_mm=0.2,          # ice film left after plowing, bonded to pavement, mm water (range 0.05 to 0.5)
    scatter_loss=0.20,        # share of salt that bounces/blows off the road            (0.05 wet - 0.30 dry)
    later_water_mm=5.0,       # extra snowmelt/rain that later rinses the road, mm        (range 0 to 20)
    unsalted_share=0.30,      # share of drain water from unsalted surfaces (lawns, roofs) (range 0 to 0.7)
    baseline_cl_mg_L=50,      # chloride already in drain water before the storm          (range 10 to 500)
    water_temp_C=1.0,         # temperature of the runoff where the probe sits            (range 0 to 5)
    probe_max_mS_cm=20.0,     # highest conductivity the DFRobot DFR0300 (K=1) can read
    probe_noise=0.03,         # random probe error, as a fraction of the reading (3%)
    temp_probe_error_C=0.5,   # DS18B20 temperature probe accuracy
    salt_price_per_ton=90,    # US$ per short ton of road salt, delivered                 (range 60 to 130)
    seed=1,                   # fixes the random sensor noise so runs are repeatable
)

# ---------------- Fixed science (reference-table values, not assumptions) ----------------
CL_SHARE = 35.45 / 58.44                       # chloride is 60.7% of salt (NaCl) by mass
LANE_MILE_M2 = 1609.34 * 3.658                 # one 12-ft lane, one mile long = 5,887 m^2
LB_PER_LANE_MILE = LANE_MILE_M2 / 453.59       # 1 g/m^2 = 13.0 lb per lane-mile

# Freezing point of salt water vs. salt strength (CRC Handbook, NaCl). Below -21.1 C salt cannot melt ice.
BRINE_PCT = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 22, 23.3]
FREEZE_C = [0, -0.59, -1.19, -1.79, -2.41, -3.05, -3.70, -4.38, -5.08, -5.81,
            -6.56, -8.18, -9.94, -11.89, -14.04, -16.46, -19.18, -21.1]

# Conductivity of salt water at 25 C (mS/cm) vs. salt concentration (g NaCl per liter). Standard values.
NACL_G_L = [0, 0.584, 2.92, 5.84, 29.2, 58.4, 117, 234, 317]
EC25 = [0, 1.185, 5.55, 10.67, 47.0, 85.8, 145, 214, 251]
ALPHA_TRUE = 0.021    # real NaCl conductivity change per C (about 2.1% per degree)
ALPHA_ASSUMED = 0.02  # the standard 2%/C correction Saltback applies


def f_to_c(f):
    return (np.asarray(f, float) - 32) * 5 / 9


def brine_strength_needed(temp_c):
    """Salt fraction a brine must have to stay liquid at temp_c (0.0 above freezing, nan below -21.1 C)."""
    t = np.asarray(temp_c, float)
    w = np.interp(-t, -np.array(FREEZE_C), BRINE_PCT) / 100
    return np.where(t >= 0, 0.0, np.where(t < FREEZE_C[-1], np.nan, w))


def ice_melted_per_salt(temp_c):
    """Pounds of ice one pound of salt can melt at temp_c (once fully dissolved)."""
    w = brine_strength_needed(temp_c)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.isnan(w), 0.0, (1 - w) / w)


def chloride_to_conductivity(cl_mg_L, water_c, alpha=ALPHA_TRUE):
    nacl_g_L = np.asarray(cl_mg_L, float) / CL_SHARE / 1000
    return np.interp(nacl_g_L, NACL_G_L, EC25) * (1 + alpha * (water_c - 25))


def conductivity_to_chloride(ec, water_c, alpha=ALPHA_ASSUMED):
    ec25 = np.asarray(ec, float) / (1 + alpha * (water_c - 25))
    return np.interp(ec25, EC25, NACL_G_L) * CL_SHARE * 1000


def sensor(cl_true, s, rng):
    """What the DFR0300 + DS18B20 report, and the chloride Saltback works out from that reading."""
    ec = chloride_to_conductivity(cl_true, s["water_temp_C"])
    reading = ec * (1 + s["probe_noise"] * rng.standard_normal(np.shape(ec)))
    clipped = reading >= s["probe_max_mS_cm"]
    reading = np.minimum(reading, s["probe_max_mS_cm"])
    temp_read = s["water_temp_C"] + s["temp_probe_error_C"] * rng.standard_normal(np.shape(ec))
    cl_est = conductivity_to_chloride(reading, temp_read)
    cl_no_tempfix = conductivity_to_chloride(reading, 25.0)   # what you'd get with no temperature probe
    return reading, clipped, cl_est, cl_no_tempfix


def storm(dose_g_m2, s, rng):
    """Simulate one storm for each salt dose (g of salt per m^2 of road). Returns a dict of result columns."""
    dose = np.asarray(dose_g_m2, float)
    temp_c = f_to_c(s["pavement_temp_F"])
    ice = s["ice_film_mm"] * 1000                     # g of ice per m^2 (1 mm of water = 1 kg/m^2)
    w = brine_strength_needed(temp_c)

    # --- Road: how much ice the salt can melt ---
    salt_on_road = dose * (1 - s["scatter_loss"])
    if temp_c >= 0:      # above freezing the ice melts by itself
        melted = np.full_like(dose, ice)
    else:
        melted = np.minimum(ice, salt_on_road * ice_melted_per_salt(temp_c))
    if np.isnan(w):      # too cold for salt to work at all: every grain is wasted
        needed, excess = np.inf, dose
    else:
        needed = ice * w / (1 - w) / (1 - s["scatter_loss"]) if w > 0 else 0.0
        excess = np.maximum(dose - needed, 0)

    # --- Brine leaving the road: once the ice is gone, extra salt only makes the brine saltier ---
    brine_w = np.where(salt_on_road <= 0, 0.0,
                       np.where(melted < ice, w, salt_on_road / (salt_on_road + ice)))
    brine_w = np.nan_to_num(np.minimum(brine_w, 0.264))      # 26.4% = fully saturated salt water
    brine_cl = brine_w * (1 + 0.72 * brine_w) * 1e6 * CL_SHARE   # mg chloride per liter of brine

    # --- Storm drain: first rush of brine (peak) and the whole-storm average ---
    mix = 1 - s["unsalted_share"]
    peak_cl = brine_cl * mix + s["baseline_cl_mg_L"]
    water_L = (ice + s["later_water_mm"] * 1000) / 1000 / mix       # liters reaching the drain per m^2 of road
    avg_cl = salt_on_road * CL_SHARE * 1000 / water_L + s["baseline_cl_mg_L"]

    # --- Sensor + Saltback's estimate ---
    pk_read, pk_clip, pk_est, _ = sensor(peak_cl, s, rng)
    av_read, av_clip, av_est, av_no_fix = sensor(avg_cl, s, rng)
    salt_seen = np.maximum(av_est - s["baseline_cl_mg_L"], 0) * water_L / CL_SHARE / 1000   # g/m^2

    lb = LB_PER_LANE_MILE
    cost_per_lb = s["salt_price_per_ton"] / 2000
    return dict(
        salt_applied_lb_per_lane_mile=dose * lb,
        salt_needed_lb_per_lane_mile=np.full_like(dose, needed * lb),
        excess_pct=np.where(dose > 0, 100 * excess / np.maximum(dose, 1e-9), 0),
        ice_cleared_pct=100 * melted / ice,
        wasted_dollars_per_lane_mile=excess * lb * cost_per_lb,
        chloride_released_lb_per_lane_mile=dose * CL_SHARE * lb,
        peak_cl_true_mg_L=peak_cl, peak_reading_mS_cm=pk_read, peak_probe_maxed_out=pk_clip,
        peak_cl_measured_mg_L=pk_est,
        avg_cl_true_mg_L=avg_cl, avg_reading_mS_cm=av_read, avg_probe_maxed_out=av_clip,
        avg_cl_measured_mg_L=av_est, avg_cl_if_no_temp_probe_mg_L=av_no_fix,
        salt_seen_by_sensor_lb_per_lane_mile=salt_seen * lb,
        salt_reaching_road_lb_per_lane_mile=salt_on_road * lb,
    )
''')
open('simulate.py', 'w').write(r'''"""Run the Saltback simulation and save raw data to results/. (explain.py turns it into plain English.)

Usage:  python simulate.py              uses model.SETTINGS
        python simulate.py my.json      overrides any settings listed in my.json
"""
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import model

OUT = Path("results")

# Widely reprinted chart: pounds of ice melted by 1 lb of salt (originally from the Salt Institute).
# Used ONLY to check the model; the model itself never reads it.
PUBLISHED_ICE_PER_LB = {30: 46.3, 25: 14.4, 20: 8.6, 15: 6.3, 10: 4.9, 5: 4.1, 0: 3.7}


def run(overrides=None):
    s = {**model.SETTINGS, **(overrides or {})}
    rng = np.random.default_rng(s["seed"])
    OUT.mkdir(exist_ok=True)

    # 1) Dose sweep: the same storm salted from 0 up to 2.5x what it needs (at least 600 lb per lane-mile)
    need = model.storm([0.0], s, rng)["salt_needed_lb_per_lane_mile"][0]
    top = int(np.clip(np.ceil(2.5 * np.nan_to_num(need, posinf=0) / 100) * 100, 600, 2500))
    doses_lb = np.arange(0, top + 1, 20)
    sweep = pd.DataFrame(model.storm(doses_lb / model.LB_PER_LANE_MILE, s, rng))
    sweep.round(2).to_csv(OUT / "dose_sweep.csv", index=False)

    # 2) Temperature chart: how much salt this storm needs at each road temperature
    temps = np.arange(-5, 33, 1)
    rows = []
    for t in temps:
        r = model.storm([0.0], {**s, "pavement_temp_F": t}, rng)
        rows.append(dict(pavement_temp_F=t,
                         salt_needed_lb_per_lane_mile=r["salt_needed_lb_per_lane_mile"][0],
                         lb_ice_melted_per_lb_salt=float(model.ice_melted_per_salt(model.f_to_c(t)))))
    pd.DataFrame(rows).round(2).to_csv(OUT / "temperature_chart.csv", index=False)

    # 3) Model check against the published chart
    check = pd.DataFrame([dict(temp_F=t, published=p,
                               model=round(float(model.ice_melted_per_salt(model.f_to_c(t))), 1))
                          for t, p in PUBLISHED_ICE_PER_LB.items()])
    check["difference_pct"] = (100 * (check.model - check.published) / check.published).round(0)
    check.to_csv(OUT / "model_check.csv", index=False)

    (OUT / "settings_used.json").write_text(json.dumps(s, indent=2))
    print(f"Simulated {len(sweep)} salt doses at {s['pavement_temp_F']}F -> {OUT}/dose_sweep.csv, "
          "temperature_chart.csv, model_check.csv, settings_used.json")
    return sweep


if __name__ == "__main__":
    run(json.loads(Path(sys.argv[1]).read_text()) if len(sys.argv) > 1 else None)
''')
open('explain.py', 'w').write(r'''"""Turn the raw numbers in results/ into a plain-English report (report.txt) and a chart (chart.png).

Run simulate.py first. This file only READS the data; it never re-runs the physics.
"""
import json
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("results")
INK, INK2, GRID, BG = "#0b0b0b", "#52514e", "#e5e4df", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"


def closest(df, col, value):
    return df.iloc[(df[col] - value).abs().argmin()]


def write_report(sweep, temps, check, s):
    need = sweep.salt_needed_lb_per_lane_mile.iloc[0]
    t_c = (s["pavement_temp_F"] - 32) * 5 / 9
    twice = closest(sweep, "salt_applied_lb_per_lane_mile", 2 * need)
    salted = sweep[sweep.salt_applied_lb_per_lane_mile > 0]
    avg_err = (100 * (salted.avg_cl_measured_mg_L - salted.avg_cl_true_mg_L) / salted.avg_cl_true_mg_L).abs().mean()
    no_probe = (100 * (1 - salted.avg_cl_if_no_temp_probe_mg_L / salted.avg_cl_true_mg_L)).mean()
    maxed = int(salted.peak_probe_maxed_out.sum())
    avg_maxed = int(salted.avg_probe_maxed_out.sum())
    seen = (100 * salted.salt_seen_by_sensor_lb_per_lane_mile / salted.salt_reaching_road_lb_per_lane_mile).mean()
    worst_check = check.difference_pct.abs()[check.temp_F <= 25].max()

    L = []
    L += ["SALTBACK SIMULATION - STEP 1: ONE STORM, IN PLAIN ENGLISH", "=" * 60, ""]
    L += [f"THE STORM: road surface at {s['pavement_temp_F']}F ({t_c:.0f}C), with a {s['ice_film_mm']} mm ice film",
          f"stuck to the road after plowing. {s['scatter_loss']:.0%} of the salt bounces off the road.", ""]
    L += ["1) HOW MUCH SALT THIS STORM NEEDED",
          f"   About {need:.0f} lb per lane-mile. Any less leaves some ice; any more melts nothing extra.", ""]
    L += ["   Salt spread      Ice cleared   Salt wasted   Money wasted (per lane-mile)"]
    top = sweep.salt_applied_lb_per_lane_mile.max()
    step = 100 * -(-top // 800)          # about 6-8 table rows, in round 100s
    for dose in range(int(step), int(top) + 1, int(step)):
        r = closest(sweep, "salt_applied_lb_per_lane_mile", dose)
        L.append(f"   {r.salt_applied_lb_per_lane_mile:5.0f} lb/lane-mi   {r.ice_cleared_pct:5.0f}%       "
                 f"{r.excess_pct:5.0f}%        ${r.wasted_dollars_per_lane_mile:5.2f}")
    if need == 0:
        L += ["", "   The road is at or above freezing: the ice melts on its own, so every grain is wasted.", ""]
    else:
        L += ["", f"   At double the needed amount (~{twice.salt_applied_lb_per_lane_mile:.0f} lb), "
              f"{twice.excess_pct:.0f}% of the salt does nothing but pollute.", ""]
    if s["pavement_temp_F"] < 15:
        L += ["   Note: below about 15F salt works very slowly, so real crews often add sand or switch",
              "   to stronger chemicals. This model only knows about plain salt (for now).", ""]
    L += ["2) TEMPERATURE CHANGES EVERYTHING"]
    for t in (30, 25, 20, 15, 10):
        r = closest(temps, "pavement_temp_F", t)
        L.append(f"   At {t:2d}F: 1 lb of salt melts {r.lb_ice_melted_per_lb_salt:4.1f} lb of ice -> "
                 f"this storm needs {r.salt_needed_lb_per_lane_mile:4.0f} lb/lane-mile")
    L += ["   Same storm, very different salt needs. A fixed 'one rate for everything' habit wastes salt",
          "   on mild days and under-treats cold ones. That is why Saltback starts from pavement temperature.", ""]
    L += ["3) CAN THE CHEAP SENSOR SEE IT?",
          f"   Storm-average runoff: the probe stayed in range {len(salted) - avg_maxed} of {len(salted)} times, and",
          f"   Saltback's chloride estimate was off by {avg_err:.0f}% on average. It found {seen:.0f}% of the salt",
          "   that actually reached the road.  -> The DFR0300 works for this job.",
          f"   Without the temperature probe, readings would be {no_probe:.0f}% too low (cold water conducts",
          "   less electricity). -> The $5 DS18B20 is essential, not optional.", ""]
    L += ["4) WARNING FOR THE HARDWARE PLAN",
          f"   The first rush of brine off the road maxed out the probe in {maxed} of {len(salted)} runs (it is",
          "   far saltier than the 20 mS/cm limit). So the DFR0300 cannot measure the first flush. Options:",
          "   measure the storm-average water (works, see #3), or add a higher-range (K=10) probe later.", ""]
    L += ["5) IS THE MODEL TRUSTWORTHY?",
          f"   It was checked against the widely published 'ice melted per pound of salt' chart: within",
          f"   {worst_check:.0f}% from 0F to 25F. (Near 30F it reads higher, because tiny temperature changes",
          "   near freezing matter a lot.) The model never reads that chart; it uses freezing-point physics.", ""]
    L += ["NOT IN THE MODEL YET (coming in later steps):",
          "   - time: how fast salt works, refreezing, new snow diluting the brine",
          "   - traffic, plowing, and salt carried off by tires",
          "   - a whole winter of storms, and Saltback learning from each one",
          "   - uncertainty: what if our guesses (ice film, runoff water) are wrong?",
          "", "All inputs are typical published values, not real town data. See results/settings_used.json."]
    text = "\n".join(L)
    (OUT / "report.txt").write_text(text)
    return text


def style(ax, title, xlabel, ylabel):
    ax.set_facecolor(BG)
    ax.set_title(title, loc="left", fontsize=11, color=INK, fontweight="bold")
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=8)


def draw_chart(sweep, temps, s):
    need = sweep.salt_needed_lb_per_lane_mile.iloc[0]
    x = sweep.salt_applied_lb_per_lane_mile
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), facecolor=BG)

    ax = axes[0]
    style(ax, "A. More salt stops helping", "Salt spread (lb per lane-mile)", "Ice cleared (%)")
    ax.axvspan(need, x.max(), color=ORANGE, alpha=0.10, linewidth=0)
    ax.plot(x, sweep.ice_cleared_pct, color=BLUE, linewidth=2)
    ax.axvline(need, color=INK2, linewidth=1, linestyle="--")
    ax.text(need + 10, 45, f"Needed:\n{need:.0f} lb", color=INK, fontsize=9)
    ax.text(x.max() - 10, 8, "Wasted salt zone", color=INK2, fontsize=9, ha="right")
    ax.set_ylim(0, 105)

    ax = axes[1]
    style(ax, "B. Colder road, more salt needed", "Pavement temperature (F)", "Salt needed (lb per lane-mile)")
    ax.plot(temps.pavement_temp_F, temps.salt_needed_lb_per_lane_mile, color=BLUE, linewidth=2)
    ax.plot([s["pavement_temp_F"]], [need], "o", color=BLUE, markersize=8, markeredgecolor=BG, markeredgewidth=2)
    ax.annotate(f"This storm ({s['pavement_temp_F']}F)", (s["pavement_temp_F"], need),
                xytext=(8, 10), textcoords="offset points", color=INK, fontsize=9)

    ax = axes[2]
    style(ax, "C. What the drain sensor reports", "Salt spread (lb per lane-mile)", "Storm-average chloride (mg/L)")
    for col, color, label in [("avg_cl_true_mg_L", INK2, "True"),
                              ("avg_cl_measured_mg_L", BLUE, "Saltback estimate"),
                              ("avg_cl_if_no_temp_probe_mg_L", ORANGE, "No temperature probe")]:
        ls = "--" if label == "True" else "-"
        ax.plot(x, sweep[col], color=color, linewidth=2, linestyle=ls, label=label)
        ax.text(x.iloc[-1] + 8, sweep[col].iloc[-1], label, color=INK, fontsize=8, va="center")
    ax.legend(frameon=False, fontsize=8, loc="upper left", labelcolor=INK)
    ax.set_xlim(0, x.max() * 1.33)

    fig.tight_layout()
    fig.savefig(OUT / "chart.png", dpi=150, facecolor=BG)
    plt.close(fig)


def run():
    sweep = pd.read_csv(OUT / "dose_sweep.csv")
    temps = pd.read_csv(OUT / "temperature_chart.csv")
    check = pd.read_csv(OUT / "model_check.csv")
    s = json.loads((OUT / "settings_used.json").read_text())
    print(write_report(sweep, temps, check, s))
    draw_chart(sweep, temps, s)
    print(f"\nChart saved to {OUT}/chart.png")


if __name__ == "__main__":
    run()
''')
print('Model files ready: model.py, simulate.py, explain.py')

In [ ]:
# @title 2. Storm settings - drag the sliders, then run this cell and the ones below
import json
pavement_temp_F = 22     # @param {type:"slider", min:-5, max:32, step:1}
ice_film_mm = 0.2        # @param {type:"slider", min:0.05, max:0.5, step:0.05}
scatter_loss = 0.2       # @param {type:"slider", min:0.0, max:0.4, step:0.05}
later_water_mm = 5.0     # @param {type:"slider", min:0, max:20, step:1}
unsalted_share = 0.3     # @param {type:"slider", min:0.0, max:0.7, step:0.05}
water_temp_C = 1.0       # @param {type:"slider", min:0, max:5, step:0.5}
salt_price_per_ton = 90  # @param {type:"slider", min:60, max:130, step:5}
settings = dict(pavement_temp_F=pavement_temp_F, ice_film_mm=ice_film_mm, scatter_loss=scatter_loss,
                later_water_mm=later_water_mm, unsalted_share=unsalted_share,
                water_temp_C=water_temp_C, salt_price_per_ton=salt_price_per_ton)
json.dump(settings, open('settings.json', 'w'))
print('Settings saved:', settings)

In [ ]:
# @title 3. Run the simulation (raw data)
!python simulate.py settings.json
import pandas as pd
pd.read_csv('results/dose_sweep.csv').head(10)

In [ ]:
# @title 4. Translate into plain English + chart
!python explain.py
from IPython.display import Image, display
display(Image('results/chart.png'))

In [ ]:
# @title 5. (Optional) Download all results as a zip
!zip -qr saltback_results.zip results
from google.colab import files
files.download('saltback_results.zip')